# Kaggle SFT runner (thin): LR sweep on sft_5k

For each LR in `LRS`: **train** (Unsloth QLoRA, 3B, one T4, fp16) → push adapter + checkpoints to the HF Hub, curves to W&B → **merge** the adapter into the 16-bit base → **vLLM eval on dev** → log dev metrics to the same W&B run → zip.

Settings: **Accelerator = GPU T4 x2**, **Internet = on**. **Required secrets** (Add-ons → Secrets): `HF_TOKEN` (write access), `WANDB_API_KEY`.

Environments:
- **System Python = the eval env**, exactly as validated in `kaggle_eval.ipynb` (`requirements-gpu.txt` on top of the Kaggle image).
- **Training runs in a venv** (`/tmp/trainenv`, `--system-site-packages`). Unsloth's dependencies are installed into the venv and don't touch the eval env (known `datasets` / `huggingface-hub` conflict).

Plan: session 1 is `LRS = [2e-4]` (bring-up + timing), session 2 is `[1e-4, 5e-4]`. Every shell command goes through `run()`, which raises on a nonzero exit. Use *Run All*.

In [ ]:
# --- run settings ---
REPO = "https://github.com/twiggytom/tiny-gambits.git"
COMMIT = "main"            # pin to a commit hash for reported runs
LRS = [2e-4]               # session 1: [2e-4]; session 2: [1e-4, 5e-4]
SEED = 0
SPLIT = "sft_5k"
CONFIG = "configs/sft_pilot.yaml"
GPU = "0"
RESUME = False             # True: continue the Hub repo's last-checkpoint/ (killed session)

import json, os, shutil, subprocess, time
T_START = time.time()

def run(cmd: str) -> float:
    """Run a shell command, stream its output, raise if it fails. Returns wall-clock seconds."""
    print("$", cmd, flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    return time.time() - t

In [ ]:
run("nvidia-smi")

In [ ]:
os.chdir("/kaggle/working")
run(f"rm -rf tiny-gambits && git clone -q {REPO} tiny-gambits")
os.chdir("/kaggle/working/tiny-gambits")
run(f"git checkout -q {COMMIT} && git log -1 --format='%H %s'")
COMMIT_HASH = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

In [ ]:
# Eval env (system Python): same as kaggle_eval.ipynb.
run("pip install -q -r requirements-gpu.txt")
run("pip freeze > /kaggle/working/pip_freeze_eval.txt")
# Training env: venv on top of the system packages; unsloth's deps go into the venv only.
# --without-pip: Kaggle's Python has no ensurepip, so `python -m venv` fails without it. The venv's
# interpreter still sees the system pip (system site-packages), and pip run by the venv interpreter
# installs into the venv.
TRAIN_PY = "/tmp/trainenv/bin/python"
run("rm -rf /tmp/trainenv && python -m venv --system-site-packages --without-pip /tmp/trainenv")
run(f"{TRAIN_PY} -m pip --version")
run(f"{TRAIN_PY} -m pip install -q -r requirements-train.txt")
run(f"{TRAIN_PY} -m pip freeze > /kaggle/working/pip_freeze_train.txt")
run(f"{TRAIN_PY} -c \"import unsloth, torch, transformers, peft, trl; print('train env:', 'unsloth', unsloth.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, 'trl', trl.__version__); assert unsloth.__file__.startswith('/tmp/trainenv'), unsloth.__file__\"")
run("python -c \"import vllm, torch, transformers; print('eval env:', 'vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__)\"")
# The training install must not have changed the eval env.
run("pip freeze > /tmp/pip_freeze_eval_after.txt && diff /kaggle/working/pip_freeze_eval.txt /tmp/pip_freeze_eval_after.txt && echo 'eval env unchanged'")

In [ ]:
from kaggle_secrets import UserSecretsClient
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[name] = UserSecretsClient().get_secret(name)   # fails loudly if a secret is missing
    print(name, "loaded from Kaggle secrets")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU

In [ ]:
# Rebuild the SFT examples (downloads the pinned raw shards; ~1-2 min). Must match the local stats.
run(f"python -m src.sft_data --config configs/data.yaml --split {SPLIT} --seed 0")
stats = json.load(open(f"data/processed/{SPLIT}.stats.json"))
assert stats["n_examples"] == {"sft_5k": 12356, "sft_15k": 37026, "sft": 123301}[SPLIT], stats["n_examples"]
SETUP_SECONDS = time.time() - T_START
print(f"setup: {SETUP_SECONDS / 60:.1f} min")

In [ ]:
def run_name(lr):
    out = subprocess.run([TRAIN_PY, "-c", f"from src.train_sft import run_name; print(run_name('{SPLIT}', {lr!r}))"],
                         capture_output=True, text=True, check=True)
    return out.stdout.strip()

done = []
for lr in LRS:
    name = run_name(lr)
    merged = f"/tmp/merged/{name}"
    print(f"\n===== {name} =====", flush=True)
    resume = "--resume" if RESUME else ""
    t_train = run(f"{TRAIN_PY} -m src.train_sft --config {CONFIG} --split {SPLIT} --lr {lr} --seed {SEED} --merge {merged} {resume}")
    t_eval = run(f"python -m src.evaluate --config configs/eval.yaml --model {merged} --backend vllm --split dev --run-name {name} --seed {SEED}")
    info = json.load(open(f"outputs/{name}/train_info.json"))
    timings = {"setup_seconds": SETUP_SECONDS, "train_process_seconds": t_train,
               "merge_seconds": info["merge_seconds"], "eval_seconds": t_eval, "commit": COMMIT_HASH}
    json.dump(timings, open(f"outputs/{name}/timings.json", "w"), indent=2)
    run(f"{TRAIN_PY} -m src.wandb_log --train-info outputs/{name}/train_info.json --metrics results/{name}/dev/metrics.json --timings outputs/{name}/timings.json")
    run(f"{TRAIN_PY} -m src.project_compute --config {CONFIG} --train-info outputs/{name}/train_info.json --timings outputs/{name}/timings.json")
    shutil.rmtree(merged)   # ~6 GB each; the adapter is on the Hub
    done.append(name)

In [ ]:
# Checks + summary. T4 must train and eval in fp16; the adapter must be on the Hub.
from huggingface_hub import list_repo_files
rows = []
for name in done:
    info = json.load(open(f"outputs/{name}/train_info.json"))
    ev = json.load(open(f"results/{name}/dev/run_info.json"))
    m = json.load(open(f"results/{name}/dev/metrics.json"))
    for what, rec in [("train", info), ("eval", ev)]:
        assert "T4" not in rec["gpu"] or rec["dtype"] == "float16", f"{name} {what}: T4 must use fp16, got {rec['dtype']}"
    files = list_repo_files(info["hub_repo"])
    assert "adapter_model.safetensors" in files and any(f.startswith("last-checkpoint/") for f in files), files
    rows.append({"run": name, "steps": info["n_steps"], "loss": round(info["final_train_loss"], 4),
                 "s/step": round(info["seconds_per_step"], 2), "peak VRAM GB": round(info["peak_vram_gb"]["reserved"], 2),
                 "first_move": m["first_move_acc"], "full_solution": m["full_solution_acc"],
                 "illegal": m["illegal_rate"], "unparseable": m["unparseable_rate"]})
import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Zip for download (Output tab): results/{run}/dev + outputs/{run}/{train_info,timings}.json + pip freezes.
bundle = "/kaggle/working/bundle"
shutil.rmtree(bundle, ignore_errors=True)
for name in done:
    shutil.copytree(f"results/{name}", f"{bundle}/results/{name}")
    os.makedirs(f"{bundle}/outputs/{name}", exist_ok=True)
    for fn in ["train_info.json", "timings.json"]:
        shutil.copy(f"outputs/{name}/{fn}", f"{bundle}/outputs/{name}/{fn}")
for fn in ["pip_freeze_eval.txt", "pip_freeze_train.txt"]:
    shutil.copy(f"/kaggle/working/{fn}", f"{bundle}/{fn}")
print(shutil.make_archive(f"/kaggle/working/sft_{'_'.join(done)}", "zip", bundle))